# Phase B.3a - Classification en zero-shot et few-shot

Ce notebook intègre les approches fondées sur un modèle de langage

- **Classification en zero-shot et few-shot** :
À partir du texte d’une revue, demander à un LLM de prédire directement si l’avis est
positif ou négatif, sans (ou avec très peu de) données d’entraînement.


## 0. Préparation des modèles

Le model choisie : Llama3.2

La collection Meta Llama 3.2 de grands modèles de langage multilingues (LLM) est un ensemble de modèles génératifs pré-entraînés et optimisés pour les instructions, disponibles en tailles 1 et 3 milliards (texte en entrée/texte en sortie). Les modèles Llama 3.2 optimisés pour le texte uniquement sont conçus pour les cas d'utilisation de dialogues multilingues, notamment la recherche et la synthèse automatiques.

In [42]:
import sys
from langchain_ollama import ChatOllama
from langchain_core.callbacks.manager import CallbackManager
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

def get_local_llm(model_name="llama3.2"):
    """
    Configure et retourne une instance du modèle local via Ollama.
    """
    print(f"Connexion au modèle local '{model_name}' via Ollama...")

    try:
        llm = ChatOllama(
            model=model_name,
            temperature=0,
            callback_manager=CallbackManager([StreamingStdOutCallbackHandler()]),
        )
        
        return llm

    except Exception as e:
        print(f"\n❌ ERREUR CRITIQUE : Impossible de joindre Ollama.")
        print(f"Détails : {e}")
        print("-" * 40)
        return None

### Prendre des "reviews" pour tests

In [43]:
# import json

# reviews_file = "./reviews.json"
# sample_reviews = []
# with open(reviews_file, 'r', encoding='utf-8') as f:
#     for line in f:
#         sample_reviews.append(json.loads(line))

In [44]:
import json
import random

reviews_file = "../data/raw/yelp_academic_reviews4students.jsonl"

reviews = []
with open(reviews_file, 'r', encoding='utf-8') as f:
    for line in f:
        reviews.append(json.loads(line))

# On filtre pour exclure les notes de 3 (neutres)
filtered_reviews = [r for r in reviews if r['stars'] != 3]

# Prendre 10 reviews aléatoires parmi les filtrées
sample_reviews = random.sample(filtered_reviews, min(200, len(filtered_reviews)))


### Créations des labels

In [45]:
def get_true_label(stars):
    if stars > 3:
        return "Positif"
    else:
        return "Négatif"

---

In [46]:
results_data = []

llm = get_local_llm()

Connexion au modèle local 'llama3.2' via Ollama...


## 1. Zero-shot

In [47]:
from langchain_core.prompts import PromptTemplate

template_zero_structured = """
Tu es un expert en analyse de sentiments.
Tâche : Analyse l'avis suivant et détermine s'il est globalement positif ou négatif.
Format de réponse attendu :
CLASSIFICATION : [STRICTEMENT "Positif" ou "Négatif"]

Avis : {texte_avis}
Réponse :
"""

prompt_zero = PromptTemplate.from_template(template_zero_structured)
chain_zero = prompt_zero | llm

print("Lancement de l'analyse Zero-shot...")

for i, review in enumerate(sample_reviews, 1):
    text = review['text']
    true_label = get_true_label(review['stars']) # On récupère le vrai label
    
    response = chain_zero.invoke({"texte_avis": text})
    content = response.content
    
    # Parsing (Extraction simple basée sur le format demandé)
    label = "Inconnu"
    reasoning = "Pas d'explication trouvée"
    
    try:
        # On découpe le texte pour trouver les clés
        lines = content.split('\n')
        for line in lines:
            if "CLASSIFICATION :" in line:
                label = line.split("CLASSIFICATION :")[1].strip()
    except Exception as e:
        print(f"Erreur de parsing sur l'avis {i}: {e}")

    results_data.append({
        "id_review": i,
        "text": text,
        "mode": "Zero-shot",
        "true_label": true_label,
        "prediction": label,
        "reasoning": reasoning
    })
    
    print(f"Avis {i} traité.")
    print(response.content)


Lancement de l'analyse Zero-shot...
Avis 1 traité.
CLASSIFICATION : Négatif
Avis 2 traité.
CLASSIFICATION : Positif
Avis 3 traité.
CLASSIFICATION : Positif
Avis 4 traité.
CLASSIFICATION : Positif
Avis 5 traité.
CLASSIFICATION : Négatif
Avis 6 traité.
CLASSIFICATION : Négatif
Avis 7 traité.
CLASSIFICATION : Négatif
Avis 8 traité.
CLASSIFICATION : Négatif
Avis 9 traité.
CLASSIFICATION : Positif
Avis 10 traité.
CLASSIFICATION : Positif
Avis 11 traité.
CLASSIFICATION : Positif
Avis 12 traité.
CLASSIFICATION : Positif
Avis 13 traité.
CLASSIFICATION : Négatif
Avis 14 traité.
CLASSIFICATION : Positif
Avis 15 traité.
CLASSIFICATION : Positif
Avis 16 traité.
CLASSIFICATION : Positif
Avis 17 traité.
CLASSIFICATION : Positif
Avis 18 traité.
CLASSIFICATION : Positif
Avis 19 traité.
CLASSIFICATION : Négatif
Avis 20 traité.
CLASSIFICATION : Positif
Avis 21 traité.
CLASSIFICATION : Positif
Avis 22 traité.
CLASSIFICATION : Positif
Avis 23 traité.
CLASSIFICATION : Négatif
Avis 24 traité.
CLASSIFICATION

## 2. Few-shot

In [48]:
template_few_structured = """
Tu es un expert en analyse de sentiments.
Tâche : Analyse l'avis suivant et détermine s'il est globalement positif ou négatif.
Format de réponse attendu :
CLASSIFICATION : [STRICTEMENT "Positif" ou "Négatif"]

Exemple 1 :
Avis : "Went for lunch and found that my burger was meh.  What was obvious was that the focus of the burgers is the amount of different and random crap they can pile on it and not the flavor of the meat.  My burger patty seemed steamed and appeared to be a preformed patty, contrary to what is stated on the menu.    I can get ground beef from Kroger and make a burger that blows them out of the water."
Réponse : Négatif

Exemple 2 :
Avis : "I needed a new tires for my wife's car. They had to special order it and had it the next day, I dropped it off in the morning before work and they called a few hours later and the car was ready. It was quick and efficient, and the woman who helped me was awesome."
Réponse : Positif

Exemple 3 :
Avis : "Jim Woltman who works at Goleta Honda is 5 stars!! He is knowledgeable, helpful, and so personable.  He did a fantastic job on my Honda.  Thank you Jim!!!  And thank you Honda for having such a fabulous employee!"
Réponse : Positif

Tâche : Classifie l'avis suivant en utilisant la même logique.
Avis : {texte_avis}
Réponse :
"""

prompt_few = PromptTemplate.from_template(template_few_structured)
chain_few = prompt_few | llm

# 2. Boucle Few-shot
print("Lancement de l'analyse Few-shot...")

for i, review in enumerate(sample_reviews, 1):
    text = review['text']
    true_label = get_true_label(review['stars'])
    
    response = chain_few.invoke({"texte_avis": text})
    content = response.content
    
    # Parsing similaire
    label = "Inconnu"
    reasoning = "Pas d'explication trouvée"
    
    try:
        lines = content.split('\n')
        for line in lines:
            if "CLASSIFICATION :" in line:
                label = line.split("CLASSIFICATION :")[1].strip()
    except:
        pass

    results_data.append({
        "id_review": i,
        "text": text,
        "mode": "Few-shot",
        "true_label": true_label,
        "prediction": label,
        "reasoning": reasoning
    })
    
    print(f"Avis {i} (Few-shot) traité.")
    print(response.content)

Lancement de l'analyse Few-shot...
Avis 1 (Few-shot) traité.
CLASSIFICATION : Négatif
Avis 2 (Few-shot) traité.
CLASSIFICATION : Positif
Avis 3 (Few-shot) traité.
CLASSIFICATION : Positif
Avis 4 (Few-shot) traité.
CLASSIFICATION : Positif
Avis 5 (Few-shot) traité.
CLASSIFICATION : Négatif
Avis 6 (Few-shot) traité.
CLASSIFICATION : Négatif
Avis 7 (Few-shot) traité.
CLASSIFICATION : Négatif
Avis 8 (Few-shot) traité.
CLASSIFICATION : Négatif
Avis 9 (Few-shot) traité.
CLASSIFICATION : Positif
Avis 10 (Few-shot) traité.
CLASSIFICATION : Positif
Avis 11 (Few-shot) traité.
CLASSIFICATION : Positif
Avis 12 (Few-shot) traité.
CLASSIFICATION : Positif
Avis 13 (Few-shot) traité.
CLASSIFICATION : Négatif
Avis 14 (Few-shot) traité.
CLASSIFICATION : Positif
Avis 15 (Few-shot) traité.
CLASSIFICATION : Positif
Avis 16 (Few-shot) traité.
CLASSIFICATION : Positif
Avis 17 (Few-shot) traité.
CLASSIFICATION : Positif
Avis 18 (Few-shot) traité.
CLASSIFICATION : Positif
Avis 19 (Few-shot) traité.
CLASSIFICAT

## 3. Comparatif

In [49]:
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Préparation des données
df_results = pd.DataFrame(results_data)

# ========== TABLEAU DE COMPARAISON =========
print("\n\n📋 TABLEAU COMPARATIF - AVIS ET PRÉDICTIONS")
print("="*50)

# Créer un tableau pivot avec avis, true_label et prédictions par mode
comparison_data = []

for review_id in df_results['id_review'].unique():
    row_data = {'id_review': review_id}
    
    # Récupérer l'avis et le vrai label
    review_rows = df_results[df_results['id_review'] == review_id]
    if len(review_rows) > 0:
        row_data['text'] = review_rows.iloc[0]['text'][:100] + "..."  # Limiter à 100 caractères
        row_data['true_label'] = review_rows.iloc[0]['true_label']
    
    # Ajouter les prédictions pour chaque mode
    for mode in ['Zero-shot', 'Few-shot']:
        pred = df_results[(df_results['id_review'] == review_id) & (df_results['mode'] == mode)]
        if len(pred) > 0:
            row_data[f'pred_{mode}'] = pred.iloc[0]['prediction']
        else:
            row_data[f'pred_{mode}'] = 'N/A'
    
    comparison_data.append(row_data)

df_comparison = pd.DataFrame(comparison_data)

# Réorganiser les colonnes
cols_order = ['id_review', 'text', 'true_label', 'pred_Zero-shot', 'pred_Few-shot']
df_comparison = df_comparison[cols_order]

# Renommer les colonnes pour un meilleur affichage
df_comparison_display = df_comparison.copy()
df_comparison_display.columns = ['ID', 'Aperçu de l\'avis', 'Vrai label', 'Prédiction Zero-shot', 'Prédiction Few-shot']

display(df_comparison_display)

# ---------------------------------------------------------------------------------------

# Filtrer les labels "Inconnu" et les prédictions invalides
df_results_clean = df_results[
    (df_results['prediction'] != 'Inconnu') & 
    (df_results['prediction'].isin(['Positif', 'Négatif']))
].copy()

# Liste des modes à évaluer (few-shot | zero-shot)
modes = df_results_clean['mode'].unique()

print("📊 ANALYSE DES PERFORMANCES")
print("="*50)

for mode in modes:
    df_mode = df_results_clean[df_results_clean['mode'] == mode]

    print(f"\n▶ Rapport détaillé pour le mode : {mode}")
    print("-" * 40)
    print(classification_report(df_mode['true_label'], df_mode['prediction']))





📋 TABLEAU COMPARATIF - AVIS ET PRÉDICTIONS


,ID,Aperçu de l'avis,Vrai label,Prédiction Zero-shot,Prédiction Few-shot
0,1,I love Vietnamese food but their pho was as ba...,Négatif,Négatif,Négatif
1,2,absolutely amazing. good service and great foo...,Positif,Positif,Positif
2,3,Wow!!! I love this place! The burger was delic...,Positif,Positif,Positif
3,4,Great food. Amazing sevice with very friendly ...,Positif,Positif,Positif
4,5,"AWFUL! Wrong food was delivered. Dirty table, ...",Négatif,Négatif,Négatif
...,...,...,...,...,...
195,196,Been to this place once before and didn't have...,Négatif,Négatif,Négatif
196,197,I am beyond thankful that Irina was not only a...,Positif,Positif,Positif
197,198,I would give 0 stars. Don't trust their paymen...,Négatif,Négatif,Négatif
198,199,"My family and I (grandmother,mother,aunt and c...",Positif,Positif,Positif


📊 ANALYSE DES PERFORMANCES

▶ Rapport détaillé pour le mode : Zero-shot
----------------------------------------
              precision    recall  f1-score   support

     Négatif       0.86      1.00      0.92        42
     Positif       1.00      0.96      0.98       157

    accuracy                           0.96       199
   macro avg       0.93      0.98      0.95       199
weighted avg       0.97      0.96      0.97       199


▶ Rapport détaillé pour le mode : Few-shot
----------------------------------------
              precision    recall  f1-score   support

     Négatif       0.89      1.00      0.94        42
     Positif       1.00      0.97      0.98       158

    accuracy                           0.97       200
   macro avg       0.95      0.98      0.96       200
weighted avg       0.98      0.97      0.98       200

